Invece di costruire la rete da zero, carica ResNet50V2 pre-addestrata su ImageNet
- Utilizza un layer keras.layer.Resizing(128,128) come primo strato per adattare le immagini di CIFAR-10 (32x32) alla risoluzione minima consigliata per ResNet50
- Blocca i pesi della base pre-addestrata (trainable=false)
- Sostituisci l'ultimo layer di classificazione con uno adatto alle 10 classi di CIFAR.
- Confronta l'accuratezza finale dopo 2 epoche rispetto al modello custom costruito nella lezione

In [ ]:
import os

# 1. DEFINIZIONE DEL MOTORE DI CALCOLO (Best Practice 2026)
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import torch 
from sklearn.metrics import confusion_matrix # Import aggiunto per la verifica finale

# 2. CARICAMENTO E PREPARAZIONE DATI
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0

# 3. ARCHITETTURA ESTREMA (BLOCCHI CONV-CONV-POOL)
def build_mnist_pro_model():
    inputs = keras.Input(shape=(28, 28, 1))
    
    # MODIFICA 1: Data Augmentation On-the-Fly (per simulare scrittura "sporca")
    x = keras.layers.RandomRotation(0.1)(inputs)
    x = keras.layers.RandomZoom(0.1)(x)
    
    # Blocco 1: MODIFICA 2 (Sperimentazione: aumento filtri a 64 per catturare più varianza)
    x = keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = keras.layers.BatchNormalization()(x) 
    x = keras.layers.MaxPooling2D((2, 2))(x)
    x = keras.layers.Dropout(0.2)(x)

    # Blocco 2: Mid-level features
    x = keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.MaxPooling2D((2, 2))(x)
    x = keras.layers.Dropout(0.3)(x)

    # Blocco 3: High-level features
    x = keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = keras.layers.BatchNormalization()(x)
    
    x = keras.layers.GlobalAveragePooling2D()(x)
    
    # Classificatore finale
    x = keras.layers.Dense(128, activation="relu")(x)
    x = keras.layers.Dropout(0.4)(x)
    outputs = keras.layers.Dense(10, activation="softmax")(x)
    
    return keras.Model(inputs, outputs)

model = build_mnist_pro_model()

# 4. COMPILAZIONE
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 5. CALLBACKS
callbacks = [
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("mnist_pro_augmented.keras", save_best_only=True)
]

# 6. TRAINING LOOP
print("[INFO] Addestramento MNIST Pro con Data Augmentation...")
history = model.fit(
    x_train, y_train,
    epochs=50, 
    batch_size=128,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1
)

# 7. VALUTAZIONE E MODIFICA 3: MATRICE DI CONFUSIONE
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\n[RISULTATO] Accuratezza sul Test Set: {acc*100:.2f}%")

# Calcolo Matrice di Confusione
y_pred_probs = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

# Teoria: La matrice evidenzia se il modello "soffre" ancora su coppie ambigue (es: 4 vs 9)
cm = confusion_matrix(y_test, y_pred)
print("\n[VERIFICA] Matrice di Confusione:")
print(cm)